In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

base_temp = 11.5
amplitude = 13
city = "Одеса"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)
print(climate.head(10))

   місто   рік  місяць  температура
0  Одеса  2021       1         -1.0
1  Одеса  2021       2          0.1
2  Одеса  2021       3          5.6
3  Одеса  2021       4         13.0
4  Одеса  2021       5         17.8
5  Одеса  2021       6         22.5
6  Одеса  2021       7         26.1
7  Одеса  2021       8         23.5
8  Одеса  2021       9         17.5
9  Одеса  2021      10         12.0


**1**

In [2]:
res_task1 = climate.groupby("рік")["температура"].agg(["mean", "min", "max"])
print(res_task1)

           mean  min   max
рік                       
2021  11.783333 -1.0  26.1
2022  10.908333 -1.7  23.6
2023  11.308333 -2.0  24.6
2024  11.166667 -1.7  24.4


**2**

In [3]:
res_task2 = climate.groupby("місяць")["температура"].agg(["mean", "std"])
print(res_task2)

          mean       std
місяць                  
1       -1.400  0.424264
2       -0.725  1.132475
3        4.100  1.023067
4       11.875  0.865544
5       17.725  0.727438
6       22.750  0.300000
7       24.500  1.116542
8       22.975  1.408013
9       17.875  1.250000
10      11.125  0.689807
11       4.975  0.618466
12      -0.275  1.135415


**3**

In [4]:
res_task3 = climate.pivot_table(
    index="місяць", 
    columns="рік", 
    values="температура", 
    aggfunc="mean"
)
print(res_task3)

рік     2021  2022  2023  2024
місяць                        
1       -1.0  -1.3  -2.0  -1.3
2        0.1  -1.7   0.4  -1.7
3        5.6   3.3   3.8   3.7
4       13.0  10.9  11.9  11.7
5       17.8  17.0  17.4  18.7
6       22.5  23.1  22.5  22.9
7       26.1  23.6  23.9  24.4
8       23.5  21.3  24.6  22.5
9       17.5  19.5  18.0  16.5
10      12.0  11.3  10.4  10.8
11       4.5   5.1   5.8   4.5
12      -0.2  -1.2  -1.0   1.3


**4**

In [5]:
def get_season(m):
    if m in [12, 1, 2]:
        return "зима"
    elif m in [3, 4, 5]:
        return "весна"
    elif m in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"

climate["сезон"] = climate["місяць"].apply(get_season)
climate["тепліше_за_середнє"] = climate["температура"] > base_temp

res_task4 = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])
print(res_task4)

тепліше_за_середнє  False  True 
сезон                           
весна                   5      7
зима                   12      0
літо                    0     12
осінь                   7      5


**5**

In [6]:
res_task5 = climate.pivot(index="місяць", columns="рік", values="температура")
print(res_task5.head())

рік     2021  2022  2023  2024
місяць                        
1       -1.0  -1.3  -2.0  -1.3
2        0.1  -1.7   0.4  -1.7
3        5.6   3.3   3.8   3.7
4       13.0  10.9  11.9  11.7
5       17.8  17.0  17.4  18.7


## Відповіді на контрольні питання ##
 * Логіка Split-Apply-Combine в groupby():
   * Split (Розбиття): Початковий DataFrame розділяється на окремі групи за унікальними значеннями вказаного стовпця (або комбінації стовпців).
   * Apply (Застосування): До кожної утвореної групи окремо застосовується аналітична чи агрегувальна функція (наприклад, mean(), sum(), std()).
   * Combine (Об'єднання): Результати обчислень для кожної групи збираються назад у єдиний підсумковий DataFrame або Series.
 * Принципова відмінність між pivot() та pivot_table():
   * pivot() вимагає абсолютної унікальності кожної пари (індекс, стовпець). Якщо є дублікати, метод падає з помилкою.
   * pivot_table() підтримує агрегацію даних. Якщо для пари (індекс, стовпець) існує кілька значень, він зводить їх до одного за допомогою заданої агрегувальної функції (параметр aggfunc, за замовчуванням mean).
 * Що показує crosstab() та порівняння з groupby(...).size():
   * pd.crosstab() обчислює таблицю сопряженности (таблицю частот) для двох або більше категоріальних факторів, показуючи кількість спільних входжень.
   * Виклик df.groupby(['cat1', 'cat2']).size() повертає тотожні за змістом дані, але у вигляді Series із мультиіндексом (довгий формат), тоді як crosstab() одразу повертає двовимірну матрицю/таблицю (широкий формат).
 * Чому дані мають бути в довгому (tidy) форматі:
   * У форматі tidy кожен рядок відповідає одному спостереженню, а кожен стовпець — одній змінній.
   * Це дозволяє векторам змінних лишатися суцільними, що є обов'язковою умовою для гнучкого розбиття даних за допомогою groupby(), виконання трансформацій pivot_table() та коректної побудови графіків. Якщо розтягнути роки чи місяці у назви стовпців (широкий формат), "рік" або "місяць" втрачають статус окремої придатної для групування змінної.